# Redrob Candidate Ranker Sandbox

Reproducible Google Colab demo for the Redrob candidate ranker. Clones the GitHub repo, downloads `candidates.jsonl` from Google Drive, runs the ranker offline on CPU, and validates the output CSV.

## 1. Install Dependencies

First, we install the CPU-only dependencies required for feature extraction and model inference. The model utilizes `numpy`, `scikit-learn`, and `xgboost` (falls back to LightGBM or Random Forest if needed).

In [1]:
# Install required libraries
!pip install -q numpy scikit-learn xgboost pandas gdown

## 2. Clone Repository

Clone the ranker code from GitHub to get `rank.py`, `validate_submission.py`, and supporting files.

In [2]:
import os

REPO_URL = "https://github.com/dhanasaur/Redrob__.git"
REPO_NAME = "Redrob__"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL}
    %cd {REPO_NAME}
else:
    print(f"{REPO_NAME} already exists, pulling changes...")
    %cd {REPO_NAME}
    !git pull

print(f"Current working directory: {os.getcwd()}")
print("Workspace files:", os.listdir("."))

Cloning https://github.com/dhanasaur/Redrob__.git...
Cloning into 'Redrob__'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 15 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 60.88 KiB | 6.09 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Redrob__
Current working directory: /content/Redrob__
Workspace files: ['.git', 'colab_sandbox.ipynb', 'candidate_schema.json', 'README.md', '.gitignore', 'requirements.txt', 'rank.py', 'sample_candidates.json', 'validate_submission.py', 'submission_metadata.yaml', 'sample_submission.csv']


## 3. Download Candidate Dataset

Download `candidates.jsonl` from Google Drive. The ranker streams this file line-by-line to keep memory usage low.

In [3]:
import gdown

DRIVE_URL = "https://drive.google.com/file/d/1rY6g0LiWjPS2lxMWssi7F5QA_Zdq79Bq/view?usp=sharing"
CANDIDATES_PATH = "candidates.jsonl"

if not os.path.exists(CANDIDATES_PATH):
    print("Downloading candidates.jsonl from Google Drive...")
    gdown.download(DRIVE_URL, CANDIDATES_PATH, fuzzy=True, quiet=False)
else:
    print(f"{CANDIDATES_PATH} already exists, skipping download.")

if os.path.exists(CANDIDATES_PATH):
    size_mb = os.path.getsize(CANDIDATES_PATH) / (1024 * 1024)
    with open(CANDIDATES_PATH, "r", encoding="utf-8") as f:
        line_count = sum(1 for _ in f)
    print(f"Ready: {CANDIDATES_PATH} ({size_mb:.1f} MB, {line_count:,} candidates)")
else:
    print("ERROR: Download failed. Check that the Drive file is shared as 'Anyone with the link'.")

Downloading...
From (original): https://drive.google.com/uc?id=1rY6g0LiWjPS2lxMWssi7F5QA_Zdq79Bq
From (redirected): https://drive.google.com/uc?id=1rY6g0LiWjPS2lxMWssi7F5QA_Zdq79Bq&confirm=t&uuid=b0d0feed-c620-4044-85f9-8dd65c0f4432
To: /content/Redrob__/candidates.jsonl
100%|██████████| 487M/487M [00:08<00:00, 60.4MB/s]


Ready: candidates.jsonl (464.7 MB, 100,000 candidates)


## 4. Run the Candidate Ranker

Next, we run the CPU-only offline ranker. It reads the candidates, extracts compact features, performs training/validation, and outputs a ranked CSV file with natural language reasoning for each candidate.

In [4]:
# Run the ranker on the full candidate pool and write a top-100 submission CSV.
!python rank.py --candidates candidates.jsonl --out submission.csv --verbose

Processed 25000 candidates...
Processed 50000 candidates...
Processed 75000 candidates...
Processed 100000 candidates...
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
Wrote submission.csv
Wrote ranker_diagnostics.md


## 5. View Top Ranked Candidates

Let's display the top 10 candidates selected by the ranker model, including their model scores and the generated reasoning.

In [5]:
import pandas as pd

if os.path.exists('submission.csv'):
    df = pd.read_csv('submission.csv')
    print(f"Ranked {len(df)} candidates. Showing Top 10:")
    pd.set_option('display.max_colwidth', None)
    display(df.head(10))
else:
    print("ERROR: submission.csv was not generated. Check the ranker output above.")

Ranked 100 candidates. Showing Top 10:


,candidate_id,rank,score,reasoning
0,CAND_0018499,1,0.990000,"Strong fit: 7.2 years as Senior Machine Learning Engineer at Zomato in Noida, Uttar Pradesh, with profile evidence for hybrid retrieval, dense retrieval, learning-to-rank, ranking pipelines, information retrieval; listed skills include Learning to Rank, Embeddings, Recommendation Systems. open to work with 15-day notice; 0.61 recruiter response rate."
1,CAND_0081846,2,0.985877,"Strong fit: 6.7 years as Lead AI Engineer at Razorpay in Jaipur, Rajasthan, with profile evidence for hybrid retrieval, dense retrieval, learning-to-rank, ranking pipelines, information retrieval; listed skills include Learning to Rank, Information Retrieval, BM25. open to work with 30-day notice; 0.73 recruiter response rate."
2,CAND_0086022,3,0.941879,"Strong fit: 5.3 years as Senior Applied Scientist at Sarvam AI in Kolkata, West Bengal, with profile evidence for hybrid retrieval, dense retrieval, learning-to-rank, ranking pipelines, retrieval; listed skills include Embeddings, Pinecone, Recommendation Systems. open to work with 0-day notice; 0.55 recruiter response rate."
3,CAND_0002025,4,0.901556,"Strong fit: 5.9 years as Senior AI Engineer at Apple in Trivandrum, Kerala, with profile evidence for hybrid retrieval, learning-to-rank, retrieval, ranking, BM25; listed skills include Recommendation Systems, FAISS, NLP. open to work with 30-day notice; 0.80 recruiter response rate; concern: not in a preferred city and no relocation signal."
4,CAND_0046064,5,0.853775,"Strong fit: 8.9 years as Senior NLP Engineer at Salesforce in Coimbatore, Tamil Nadu, with profile evidence for hybrid retrieval, dense retrieval, learning-to-rank, ranking pipelines, retrieval; listed skills include BM25, Pinecone, Elasticsearch. open to work with 30-day notice; 0.78 recruiter response rate; concern: not in a preferred city and no relocation signal."
5,CAND_0046525,6,0.835412,"Strong fit: 6.1 years as Senior Machine Learning Engineer at Genpact AI in Pune, Maharashtra, with profile evidence for hybrid retrieval, dense retrieval, learning-to-rank, ranking pipelines, information retrieval; listed skills include Information Retrieval, Qdrant, Elasticsearch. open to work; 0.88 recruiter response rate; concern: 60-day notice period is slower than ideal."
6,CAND_0071974,7,0.818683,"Strong fit: 7.8 years as Senior AI Engineer at Netflix in Vizag, Andhra Pradesh, with profile evidence for hybrid retrieval, learning-to-rank, ranking pipelines, information retrieval, retrieval; listed skills include BM25, Embeddings, Qdrant. open to work; 0.76 recruiter response rate; concern: 45-day notice period is slower than ideal."
7,CAND_0077337,8,0.817076,"Strong fit: 7.0 years as Staff Machine Learning Engineer at Paytm in Kochi, Kerala, with profile evidence for hybrid retrieval, learning-to-rank, information retrieval, retrieval, ranking; listed skills include Information Retrieval, BM25, RAG. open to work; 0.95 recruiter response rate; concern: 60-day notice period is slower than ideal."
8,CAND_0011687,9,0.805219,"Strong fit: 7.8 years as Senior NLP Engineer at Niramai in Indore, Madhya Pradesh, with profile evidence for hybrid retrieval, learning-to-rank, ranking pipelines, retrieval, ranking; listed skills include Embeddings, Weaviate, OpenSearch. open to work with 15-day notice; 0.89 recruiter response rate; concern: not in a preferred city and no relocation signal."
9,CAND_0055905,10,0.750137,"Strong fit: 8.1 years as Senior Machine Learning Engineer at Flipkart in London, with profile evidence for hybrid retrieval, dense retrieval, learning-to-rank, ranking pipelines, information retrieval; listed skills include Embeddings, Information Retrieval, OpenSearch. open to work with 30-day notice; 0.87 recruiter response rate; concern: outside India with no relocation signal."


## 6. Validate Submission Format

Run the official validator to confirm the CSV has exactly 100 rows, correct columns, and monotonically decreasing scores.

In [6]:
!python validate_submission.py submission.csv

Submission is valid.
